# Interpolation Case 02: Interpolation Denoising

This notebook uses Navigo to examine whether interpolation can smooth an anomalous myofibroblast population at E18.25 and recover coherent marker and pathway readouts along the muscle trajectory.

In this notebook you will:
1. Generate adjacent-time interpolation predictions for the muscle trajectory.
2. Compare noisy versus Navigo-derived marker dynamics across late developmental stages.
3. Test whether the denoised population recovers biologically relevant pathway enrichment at E18.25.


Import required packages and set deterministic seeds.

In [1]:
import configparser
import json
import os
import random
import subprocess
import sys
from pathlib import Path

import anndata
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Image, display
from scipy import sparse, stats
from sklearn.neighbors import KNeighborsClassifier

Define shared helpers for path resolution, dense conversion, and DEG testing.

In [2]:
def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'docs' / 'tutorials').exists() and (p / 'navigo').exists():
            return p
    raise RuntimeError('Could not locate repository root containing docs/tutorials and the navigo package.')


def to_dense(x):
    return x.toarray() if sparse.issparse(x) else np.asarray(x)


def sanitize_cell_type(cell_type: str) -> str:
    return (
        cell_type.replace('/', '_')
        .replace(' ', '_')
        .replace('(', '|')
        .replace(')', '|')
    )


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def bh_fdr(pvals: np.ndarray) -> np.ndarray:
    pvals = np.asarray(pvals, dtype=float)
    n = pvals.size
    order = np.argsort(pvals)
    ranked = pvals[order]

    adj_ranked = np.empty(n, dtype=float)
    prev = 1.0
    for i in range(n - 1, -1, -1):
        rank = i + 1
        val = min(prev, ranked[i] * n / rank)
        adj_ranked[i] = val
        prev = val

    adjusted = np.empty(n, dtype=float)
    adjusted[order] = np.clip(adj_ranked, 0.0, 1.0)
    return adjusted


def wilcoxon_deg(x_target: np.ndarray, x_other: np.ndarray, gene_names: np.ndarray) -> pd.DataFrame:
    mean_target = x_target.mean(axis=0)
    mean_other = x_other.mean(axis=0)
    logfc = np.log2((mean_target + 1e-9) / (mean_other + 1e-9))

    pvals = np.ones(x_target.shape[1], dtype=float)
    scores = np.zeros(x_target.shape[1], dtype=float)

    for j in range(x_target.shape[1]):
        a = x_target[:, j]
        b = x_other[:, j]

        if np.allclose(a, a[0]) and np.allclose(b, b[0]) and np.isclose(a[0], b[0]):
            pvals[j] = 1.0
            scores[j] = 0.0
            continue

        res = stats.mannwhitneyu(a, b, alternative='two-sided', method='asymptotic')
        pvals[j] = res.pvalue
        scores[j] = res.statistic

    pvals_adj = bh_fdr(pvals)
    df = pd.DataFrame({
        'names': gene_names,
        'scores': scores,
        'logfoldchanges': logfc,
        'pvals': pvals,
        'pvals_adj': pvals_adj,
    })
    return df.sort_values(['pvals_adj', 'pvals', 'logfoldchanges'], ascending=[True, True, False]).reset_index(drop=True)


def normalize_to_x(adata: anndata.AnnData) -> anndata.AnnData:
    adata_m = np.concatenate([to_dense(adata.layers['Ms']), to_dense(adata.layers['Mu'])], axis=1)
    data = (adata_m - adata_m.min(axis=0)) / (adata_m.max(axis=0) - adata_m.min(axis=0) + 1e-7)
    n_vars = adata.n_vars
    adata.X = data[:, :n_vars] + data[:, n_vars:]
    return adata



def sorted_days(day_values) -> list[str]:
    def day_to_float(day):
        day = str(day)
        return 19.5 if day == 'P0' else float(day[1:])

    return [str(day) for day in sorted({str(day) for day in day_values}, key=day_to_float)]


Set paths and runtime configuration for this interpolation case.

In [3]:
repo_root = find_repo_root(Path.cwd())
tutorials_root = repo_root / 'docs' / 'tutorials'
section_dir = tutorials_root

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

data_dir = repo_root / 'data' / 'interpolation'
checkpoints_dir = repo_root / 'checkpoints' / 'interpolation'
marker_script = repo_root / 'navigo' / 'interpolation_case_panel_marker_gene.py'
pathway_script = repo_root / 'navigo' / 'interpolation_case_panel_pathway_enrichment.py'
proportion_script = repo_root / 'navigo' / 'interpolation_case_panel_proportion_count.py'

FULL_DATA = data_dir / 'mouse_embryogenesis_aggregated_full_hvg_4000.h5ad'
CKPT = checkpoints_dir / 'myofibroblast_imputation_checkpoint_round6.pth'
MSIGDB_PATH = data_dir / 'msigdb_mouse_v2025_1.json'
CT_TO_TRAJ_JSON = data_dir / 'ct_to_trajectory.json'

NOTEBOOK_OUT = tutorials_root / 'outputs' / 'interpolation_myofibroblasts'
PRED_DIR = NOTEBOOK_OUT / '00_model_inference'
DEG_DIR = NOTEBOOK_OUT / '00_deg_results'
case_dir = NOTEBOOK_OUT / '00_case_cache'
TABLE_DIR = NOTEBOOK_OUT / '01_tables'
FIG_DIR = NOTEBOOK_OUT / '02_figures'

for p in [PRED_DIR, DEG_DIR, TABLE_DIR, FIG_DIR, case_dir]:
    p.mkdir(parents=True, exist_ok=True)

CELL_TYPE = 'Myofibroblasts'
TARGET_DAY = 18.25
START_DAY = 8.5
STEP = 0.25
SOURCE_INDICES = list(range(22, 42))

RUN_INFERENCE = True
SKIP_EXISTING_INFERENCE = True
KNN_TRAIN_MAX_CELLS = 80000
N_NEIGHBORS = 20

RUN_DEG = True
SKIP_EXISTING_DEG = True
MIN_CELLS_PER_GROUP = 3

ct_file = sanitize_cell_type(CELL_TYPE)
point = int(round((TARGET_DAY - START_DAY) / STEP))
marker_csv = case_dir / f'{ct_file}_marker_genes_t{point}.csv'
shared_csv = case_dir / f'{ct_file}_shared_pathways.csv'
prop_csv = TABLE_DIR / '01_trajectory_proportion_table.csv'

analysis_cmds = [
    [
        sys.executable,
        str(marker_script),
        '--input_data', str(FULL_DATA),
        '--pred_dir', str(PRED_DIR),
        '--deg_dir', str(DEG_DIR),
        '--cell_type', CELL_TYPE,
        '--target_day', str(TARGET_DAY),
        '--start_day', str(START_DAY),
        '--step', str(STEP),
        '--output_csv', str(marker_csv),
    ],
    [
        sys.executable,
        str(pathway_script),
        '--input_data', str(FULL_DATA),
        '--pred_dir', str(PRED_DIR),
        '--deg_dir', str(DEG_DIR),
        '--msigdb_path', str(MSIGDB_PATH),
        '--cell_type', CELL_TYPE,
        '--target_day', str(TARGET_DAY),
        '--start_day', str(START_DAY),
        '--step', str(STEP),
        '--output_dir', str(case_dir),
    ],
    [
        sys.executable,
        str(proportion_script),
        '--pred_dir', str(PRED_DIR),
        '--ct_to_trajectory_json', str(CT_TO_TRAJ_JSON),
        '--cell_type', CELL_TYPE,
        '--target_day', str(TARGET_DAY),
        '--start_day', str(START_DAY),
        '--step', str(STEP),
        '--output_csv', str(prop_csv),
    ],
]


## Step 1: Run temporal interpolation inference for the muscle trajectory
Generate adjacent-time prediction files `pred_t{source}_to_t{source+1}.h5ad` with `checkpoint-6`, then use those interpolated populations as the denoised myofibroblast readout for downstream analysis.

In [4]:
expected_files = [PRED_DIR / f'pred_t{i}_to_t{i+1}.h5ad' for i in SOURCE_INDICES]
existing_count = sum(int(p.exists()) for p in expected_files)

if RUN_INFERENCE and not (SKIP_EXISTING_INFERENCE and existing_count == len(expected_files)):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = create_model(device)
    model.load_state_dict(torch.load(CKPT, map_location=device))
    model.eval()

    flow = Navigo(model=model, num_steps=10, device=device)

    data, time_label, adata, _ = load_and_preprocess_data(FULL_DATA)
    X_train_full = (data[:, :adata.n_vars] + data[:, adata.n_vars:]).cpu().numpy()
    y_train = adata.obs['cell_type'].values

    if KNN_TRAIN_MAX_CELLS and len(y_train) > KNN_TRAIN_MAX_CELLS:
        rng = np.random.default_rng(42)
        keep_idx = rng.choice(len(y_train), size=KNN_TRAIN_MAX_CELLS, replace=False)
        X_train = X_train_full[keep_idx]
        y_train_fit = y_train[keep_idx]
    else:
        X_train = X_train_full
        y_train_fit = y_train

    knn = KNeighborsClassifier(n_neighbors=N_NEIGHBORS)
    knn.fit(X_train, y_train_fit)

    all_time_set = set(float(x) for x in np.sort(np.unique(time_label.numpy())).tolist())
    for i in SOURCE_INDICES:
        source_time = float(i)
        target_time = float(i + 1)
        out_file = PRED_DIR / f'pred_t{i}_to_t{i+1}.h5ad'

        if SKIP_EXISTING_INFERENCE and out_file.exists():
            continue
        if source_time not in all_time_set or target_time not in all_time_set:
            continue

        source_mask = time_label == source_time
        z0 = data[source_mask].to(device)
        t0 = torch.tensor([source_time] * z0.shape[0]).to(device)
        t1 = torch.tensor([target_time] * z0.shape[0]).to(device)

        pred_forward = flow.sample_ode_time_interval(z_full=z0, t_start=t0, t_end=t1, N=100)
        pred_forward = pred_forward[:, :adata.n_vars] + pred_forward[:, adata.n_vars:]
        y_pred = knn.predict(pred_forward)

        pred_adata = anndata.AnnData(X=pred_forward)
        pred_adata.obs['predicted_cell_type'] = y_pred
        pred_adata.obs['source_time'] = source_time
        pred_adata.obs['target_time'] = target_time
        pred_adata.write_h5ad(out_file)

prediction_summary = {
    'prediction_files_available': sum(int(p.exists()) for p in expected_files),
    'prediction_files_expected': len(expected_files),
}
prediction_summary

{'prediction_files_available': 20, 'prediction_files_expected': 20}

## Step 2: Compute day-specific DEG tables for Myofibroblasts
For each developmental day in Myofibroblasts, we compare the focal day against all other days using the Mann-Whitney test with Benjamini-Hochberg correction. These DEG tables support the marker and pathway summaries used in the denoising analysis.

In [5]:
ct_file = sanitize_cell_type(CELL_TYPE)
existing_deg = sorted(DEG_DIR.glob(f'{ct_file}_E*_deg.csv'))

if RUN_DEG:
    adata_backed = anndata.read_h5ad(FULL_DATA, backed='r')
    obs = adata_backed.obs.copy()
    mask = (obs['cell_type'].astype(str) == CELL_TYPE).values
    adata_ct = adata_backed[mask].to_memory()
    adata_ct = normalize_to_x(adata_ct)

    x_ct = to_dense(adata_ct.X)
    gene_names = np.asarray(adata_ct.var_names.astype(str))
    days = sorted_days(adata_ct.obs['day'])
    day_values = adata_ct.obs['day'].astype(str).values

    for day in days:
        out_file = DEG_DIR / f'{ct_file}_{day.replace("P0", "E19.5")}_deg.csv'
        if SKIP_EXISTING_DEG and out_file.exists():
            continue

        day_mask = day_values == day
        other_mask = ~day_mask
        n_target = int(day_mask.sum())
        n_other = int(other_mask.sum())
        if n_target < MIN_CELLS_PER_GROUP or n_other < MIN_CELLS_PER_GROUP:
            continue

        df_deg = wilcoxon_deg(x_ct[day_mask], x_ct[other_mask], gene_names)
        df_deg.to_csv(out_file, index=False)

existing_deg_after = sorted(DEG_DIR.glob(f'{ct_file}_E*_deg.csv'))
{'deg_tables_available': len(existing_deg_after)}

{'deg_tables_available': 36}

## Step 3: Generate summary tables
Build the proportion, marker-gene, and pathway-enrichment tables used to interpret the myofibroblast denoising result.

In [6]:
for cmd in analysis_cmds:
    proc = subprocess.run(cmd, check=True, capture_output=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError(proc.stderr)

for path in [marker_csv, shared_csv, prop_csv]:
    if not path.exists():
        raise FileNotFoundError(path)

prop_display_df = pd.read_csv(prop_csv)
keep_cols = [col for col in ['trajectory', 'cell_type', 'count', 'ratio', 'target_day'] if col in prop_display_df.columns]
prop_display_df[keep_cols]


,trajectory,cell_type,count,ratio,target_day
0,Muscle_cells,Muscle progenitor cells,0,0.000000,18.25
1,Muscle_cells,Muscle progenitor cells (Prdm1+),0,0.000000,18.25
2,Muscle_cells,Myoblasts,230,0.541176,18.25
3,Muscle_cells,Myofibroblasts,170,0.400000,18.25
4,Muscle_cells,Myotubes,25,0.058824,18.25


In [7]:
from navigo.interpolation_case_render_end_to_end_figures import render_all_panels

summary = render_all_panels(
    input_data=FULL_DATA,
    pred_dir=PRED_DIR,
    ct_to_trajectory_json=CT_TO_TRAJ_JSON,
    cell_type=CELL_TYPE,
    target_day=TARGET_DAY,
    start_day=START_DAY,
    step=STEP,
    case_dir=case_dir,
    output_root=NOTEBOOK_OUT,
)

j_fig = Path(summary['paths']['panel_j'])
k_fig = Path(summary['paths']['panel_k'])
l_fig = Path(summary['paths']['panel_l'])
fig_paths = [j_fig, k_fig, l_fig]
titles = ['Trajectory proportion', 'Marker expression', 'Pathway enrichment']

fig, axes = plt.subplots(3, 1, figsize=(10, 14))
for ax, img_path, title in zip(axes, fig_paths, titles):
    ax.imshow(plt.imread(img_path))
    ax.set_title(title, fontsize=12)
    ax.axis('off')
plt.tight_layout()
plt.show()


## How to read the results
- **Proportion continuity**: Compare noisy and Navigo-derived proportions across the muscle trajectory; the key question is whether the apparent E18.25 irregularity in myofibroblast abundance is pulled back toward the trend suggested by adjacent stages.
- **Marker continuity**: Marker continuity is evaluated with `Cdkn1c`, `Synpo2`, `Myf6`, `Tnnc2`, `Myog`, and `Aldoa`; improved agreement with neighboring stages supports successful denoising.
- **Pathway recovery**: Pathway analysis tests whether the denoised population recovers coherent enrichment for skeletal muscle, metabolic, stress-response, and movement-associated programs that are weakened or distorted in the noisy E18.25 population.
